# Activity: Forecasting Rain with Multiplicative Weights
In this graded activity, we use the multiplicative weights algorithm to predict daily rain by aggregating the advice of several forecast services. We grade the result against the best service in hindsight and against the algorithm's theoretical regret bound.

> __Learning Objectives.__
>
> By the end of this activity, you will be able to:
>
> * __Aggregate expert advice with multiplicative weights:__ Maintain a probability distribution over forecast services and update it from each day's cost so the algorithm follows the better services over time.
> * __Measure regret against the best expert in hindsight:__ Compute the algorithm's cumulative loss and the best single service's cumulative loss, and take their difference.
> * __Verify the regret bound:__ Confirm the realized regret stays below the proven bound and that the average regret per round shrinks as the horizon grows.

Let's get started!
___

## Background: prediction from expert advice
The multiplicative weights algorithm follows the advice of $N$ experts and reweights them by how well they predict. We hold a probability distribution over the experts, follow it, observe the outcome, and shift weight toward the experts who were right.

> Each round $t$ the algorithm holds a distribution $\mathbf{p}^{(t)}$ over the $N$ experts, where $p_i^{(t)} = w_i^{(t)}/\sum_j w_j^{(t)}$. After the outcome is revealed, each expert receives a cost $m_i^{(t)} = -1$ if it was correct and $m_i^{(t)} = +1$ if it was incorrect, and the weights are updated multiplicatively, $w_i^{(t+1)} = w_i^{(t)}\left(1 - \eta\, m_i^{(t)}\right)$. The regret compares the algorithm's expected loss to the best single expert in hindsight, $R(T) = \sum_{t} \mathbf{p}^{(t)}\cdot\mathbf{m}^{(t)} - \min_{i}\sum_{t} m_i^{(t)}$. With the learning rate $\eta = \sqrt{\ln N / T}$, the regret is bounded by $R(T) \le 2\sqrt{T \ln N}$.

This logic is implemented in [the `play(...)` method](src/MWA.jl) for [the `MyMultiplicativeWeightsAlgorithmModel` type](src/MWA.jl). Let's set up the forecasting problem.
___

## The rain forecasting problem
We predict whether it rains over $T = 1000$ days. Each morning we consult $N = 5$ forecast services, encode rain as $+1$ and no rain as $-1$, and follow the weighted advice of the services. At the end of the day the true weather is revealed: a correct call earns a cost of $-1$ and an incorrect call a cost of $+1$. The services differ in skill, but we do not know in advance which one is best; the algorithm sees only the forecasts and the outcomes.

> __Goal.__ Combine the $N$ services so that our cumulative loss is close to that of the best single service in hindsight, without knowing in advance which service that is.

We synthesize the data by drawing each true outcome and then each service's forecast, where a more skilled service agrees with the truth more often. Let's load the environment.
___

## Setup, Data, and Prerequisites
We include the `Include.jl` file to load the required packages and the module's local `src/` codes.

In [1]:
include("Include.jl"); # load my codes, packages, etc

## Task 1: Build the forecasting problem
Synthesize the true weather and the five services' forecasts, then construct [a `MyMultiplicativeWeightsAlgorithmModel` instance](src/MWA.jl) holding the forecasts, the outcomes, and the learning rate $\eta = \sqrt{\ln N / T}$. Use [the `build(...)` method](src/MWA.jl) and store it in `mwa_problem`. The `reliability` values shape the data only; the algorithm never sees them.

In [2]:
mwa_problem = let

    Random.seed!(1234); # reproducible data

    T = 1000;                                  # number of days
    reliability = [0.90, 0.78, 0.65, 0.50, 0.35]; # per-service chance of a correct call
    N = length(reliability);                   # number of forecast services
    p_rain = 0.40;                             # fraction of rainy days

    # draw the true weather: +1 = rain, -1 = no rain -
    outcomes = [rand() < p_rain ? 1 : -1 for t in 1:T];

    # draw each service's forecast: agrees with the truth with prob = reliability -
    forecasts = zeros(Int, T, N);
    for t in 1:T, i in 1:N
        forecasts[t, i] = (rand() < reliability[i]) ? outcomes[t] : -outcomes[t];
    end

    η = sqrt(log(N) / T); # theory-optimal learning rate

    problem = build(MyMultiplicativeWeightsAlgorithmModel, (
        η = η, forecasts = forecasts, outcomes = outcomes));
    problem; # return the problem
end;

Let's confirm the problem was built with the expected dimensions and valid data.

In [3]:
let
    @assert size(mwa_problem.forecasts) == (1000, 5)
    @assert length(mwa_problem.outcomes) == 1000
    @assert all(x -> x in (-1, 1), mwa_problem.forecasts)
    @assert all(x -> x in (-1, 1), mwa_problem.outcomes)
    @assert 0 < mwa_problem.η <= 0.5
    println("Task 1 checks passed.");
end

Task 1 checks passed.


## Task 2: Run the multiplicative weights algorithm
Pass the `mwa_problem` to [the `play(...)` method](src/MWA.jl). It forms the probability distribution over services each day, scores every service, accumulates the algorithm's expected loss, and updates the weights. Store the result dictionary in `mwa_result`.

In [4]:
mwa_result = play(mwa_problem);

The per-round distributions should be valid probability vectors and the weights should stay nonnegative. Let's verify.

In [5]:
let
    p = mwa_result["p"];
    weights = mwa_result["weights"];

    println("algorithm expected loss = ", round(mwa_result["algorithm_loss"], digits = 4));

    @assert all(isapprox.(sum(p, dims = 2), 1.0; atol = 1e-9)) # each row is a distribution
    @assert all(p .>= 0.0)                                      # valid probabilities
    @assert all(weights .>= 0.0)                                # weights stay nonnegative
    println("Task 2 checks passed.");
end

algorithm expected loss = -775.9107
Task 2 checks passed.


## Task 3: Find the best service in hindsight
To grade the algorithm we need the best single service in hindsight, i.e., the service with the lowest cumulative cost over all $T$ days. We compute it independently from the cost matrix returned by `play(...)` and confirm it matches the value the solver reported.

In [6]:
best_in_hindsight = let
    cost = mwa_result["cost"];
    cumulative = vec(sum(cost, dims = 1)); # cumulative cost per service
    best = argmin(cumulative);             # lowest cumulative cost wins

    accuracy = mean(cost[:, best] .== -1.0); # fraction of correct calls
    println("best service       = service $(best)");
    println("its cumulative cost = ", round(cumulative[best], digits = 2));
    println("its accuracy        = ", round(100*accuracy, digits = 1), " %");

    @assert best == mwa_result["best_expert"] # matches the solver
    @assert isapprox(cumulative[best], mwa_result["best_expert_loss"])
    println("Task 3 checks passed.");
    best;
end;

best service       = service 1
its cumulative cost = -818.0
its accuracy        = 90.9 %
Task 3 checks passed.


## Task 4: Verify the regret bound
The realized regret is the algorithm's cumulative loss minus the best service's cumulative loss. The theory guarantees it stays below $2\sqrt{T\ln N}$, and that the average regret per round is small. We grade the algorithm by checking both.

In [7]:
let
    T = mwa_problem.T;
    N = mwa_problem.n;
    regret = mwa_result["regret"];
    bound = 2*sqrt(T*log(N));

    println("algorithm loss     = ", round(mwa_result["algorithm_loss"], digits = 3));
    println("best service loss  = ", round(mwa_result["best_expert_loss"], digits = 3));
    println("realized regret    = ", round(regret, digits = 3));
    println("theoretical bound  = ", round(bound, digits = 3));
    println("avg regret / round = ", round(regret/T, digits = 5));

    @assert regret <= bound  # the proven guarantee holds
    @assert regret/T < 0.1   # average regret per round is small
    println("Task 4 checks passed — regret stays below the bound.");
end

algorithm loss     = -775.911
best service loss  = -818.0
realized regret    = 42.089
theoretical bound  = 80.236
avg regret / round = 0.04209
Task 4 checks passed — regret stays below the bound.


### The average regret per round vanishes
The bound $R(T) \le 2\sqrt{T\ln N}$ grows like $\sqrt{T}$, so the average regret per round $R(T)/T$ shrinks toward zero as the horizon grows. We rerun the same setup at several horizons and tabulate the regret.

In [8]:
let
    reliability = [0.90, 0.78, 0.65, 0.50, 0.35];
    N = length(reliability);
    p_rain = 0.40;
    df = DataFrame();

    for T in (250, 500, 1000, 2000)
        Random.seed!(1234);
        outcomes = [rand() < p_rain ? 1 : -1 for t in 1:T];
        forecasts = zeros(Int, T, N);
        for t in 1:T, i in 1:N
            forecasts[t, i] = (rand() < reliability[i]) ? outcomes[t] : -outcomes[t];
        end
        η = sqrt(log(N) / T);
        prob = build(MyMultiplicativeWeightsAlgorithmModel, (
            η = η, forecasts = forecasts, outcomes = outcomes));
        res = play(prob);
        push!(df, (T = T, regret = round(res["regret"], digits = 2),
            bound = round(2*sqrt(T*log(N)), digits = 2),
            avg_regret = round(res["regret"]/T, digits = 4)));
    end
    pretty_table(df;
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact));
end

 ------- --------- --------- ------------
      T    regret     bound   avg_regret 
  Int64   Float64   Float64      Float64 
 ------- --------- --------- ------------
    250     22.22     40.12       0.0889
    500     30.22     56.74       0.0604
   1000     42.09     80.24       0.0421
   2000     58.49    113.47       0.0292
 ------- --------- --------- ------------


### The forecast services
Finally, let's compare the services: each one's accuracy, its cumulative cost over the $T$ days, and the final weight the algorithm assigned to it. `Unhide` the code block below to see how we build the table.

In [9]:
let
    cost = mwa_result["cost"];
    weights = mwa_result["weights"];
    T = mwa_problem.T;
    N = mwa_problem.n;

    final_weights = weights[end, :];
    df = DataFrame();
    for i in 1:N
        push!(df, (service = "Service-$(i)",
            accuracy = round(100*mean(cost[:, i] .== -1.0), digits = 1),
            cumulative_cost = sum(cost[:, i]),
            final_weight = round(final_weights[i], digits = 4)));
    end
    pretty_table(df;
        backend = :text,
        table_format = TextTableFormat(borders = text_table_borders__compact));
end

 ----------- ---------- ----------------- --------------
    service   accuracy   cumulative_cost   final_weight 
     String    Float64           Float64        Float64 
 ----------- ---------- ----------------- --------------
  Service-1       90.9            -818.0            1.0
  Service-2       76.4            -528.0            0.0
  Service-3       66.4            -328.0            0.0
  Service-4       52.7             -54.0            0.0
  Service-5       31.7             366.0            0.0
 ----------- ---------- ----------------- --------------


___
## Summary
In this activity, we aggregated several forecast services with the multiplicative weights algorithm, then validated the result against the best service in hindsight and the algorithm's regret bound.

> __Key Takeaways:__
>
> * __Learning from expert advice:__ Maintaining a probability distribution over services and updating it multiplicatively from each day's cost lets the algorithm shift weight toward the services that predict well.
> * __Regret against the best expert:__ The regret is the algorithm's cumulative loss minus the best single service's cumulative loss, which measures how much the algorithm gives up for not knowing the best service in advance.
> * __The bound holds and tightens:__ The realized regret stays below the theoretical bound, and because that bound grows with the square root of the horizon, the average regret per round shrinks toward zero as the number of rounds grows.

The multiplicative weights algorithm tracks the best forecast service in hindsight without knowing in advance which service is best. This makes it a foundation for online learning in dynamic, uncertain environments where the best strategy is revealed only over time.
___